In [8]:
#IMPORTACION DE LIBERIAS

# Manejo de datos
import numpy as np
import pandas as pd

# Visualizacion
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



In [9]:
# Cargar el checkpoint del notebook anterior
df_modelo = pd.read_parquet('datos_modelo_limpio.parquet')

# Para asegurar que nuestro orden cronológico esté intacto
# En series de tiempo, si el orden falla, el shift arruina todo.
df_modelo = df_modelo.sort_values(['purchase_year', 'purchase_month', 'product_category_name'])
df_modelo.head()

,purchase_year,purchase_month,product_category_name,target_demanda,precio_promedio,es_black_friday,es_fin_de_semana,es_envio_gratis
0,2017,1,agro_industria_e_comercio,3,21.990000,0,0.000000,0.0
1,2017,1,alimentos,2,21.410000,0,0.500000,0.0
2,2017,1,automotivo,31,155.484839,0,0.096774,0.0
3,2017,1,bebes,40,155.447000,0,0.100000,0.0
4,2017,1,beleza_saude,82,150.810244,0,0.170732,0.0


# FEATURE ENGINEERING


In [10]:
# (Lag 1) Dar memoria a corto plazo
df_modelo['ventas_mes_pasado'] = df_modelo.groupby('product_category_name')['target_demanda'].shift(1)

# Comprobación
columnas_ver = ['purchase_year', 'purchase_month', 'target_demanda', 'ventas_mes_pasado']
print(df_modelo[df_modelo['product_category_name'] == 'beleza_saude'][columnas_ver].head(5))

# Eliminamos exclusivamente las filas que se quedaron sin 'ventas_mes_pasado'
df_modelo = df_modelo.dropna(subset=['ventas_mes_pasado'])
df_modelo = df_modelo.reset_index(drop=True)


     purchase_year  purchase_month  target_demanda  ventas_mes_pasado
4             2017               1              82                NaN
47            2017               2             154               82.0
100           2017               3             208              154.0
151           2017               4             180              208.0
210           2017               5             284              180.0


In [11]:
# Medias Móviles (Rolling Means) Visión de tendencia a mediano plazo

# Asegurar el orden estrictamente cronológico por categoría
df_modelo = df_modelo.sort_values(['product_category_name', 'purchase_year', 'purchase_month'])

# Calculamos el rolling sobre 'ventas_mes_pasado' para evitar el Data Leakage
df_modelo['tendencia_3_meses'] = df_modelo.groupby('product_category_name')['ventas_mes_pasado'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)
    # Un bloque de 3 meses en total
    # Evitar nulos, calculando con los periodos disponibles
print(df_modelo[df_modelo['product_category_name'] == 'beleza_saude'].head(5))

     purchase_year  purchase_month product_category_name  target_demanda  \
4             2017               2          beleza_saude             154   
44            2017               3          beleza_saude             208   
92            2017               4          beleza_saude             180   
145           2017               5          beleza_saude             284   
203           2017               6          beleza_saude             254   

     precio_promedio  es_black_friday  es_fin_de_semana  es_envio_gratis  \
4         143.246234                0          0.259740              0.0   
44        123.329087                0          0.158654              0.0   
92        123.090778                0          0.338889              0.0   
145       163.169225                0          0.214789              0.0   
203       124.711850                0          0.240157              0.0   

     ventas_mes_pasado  tendencia_3_meses  
4                 82.0          82.000000 

In [12]:
# Calendario Cíclico - Codificación Cíclica usando Trigonometría

# Numeros en circulo, asi el ciclo se repite y 12 no sera mayor que 1

# Usamos 2 * pi para crear el círculo matemático completo
df_modelo['mes_seno'] = np.sin(2 * np.pi * df_modelo['purchase_month'] / 12)
df_modelo['mes_coseno'] = np.cos(2 * np.pi * df_modelo['purchase_month'] / 12)

print(df_modelo.head())
print(df_modelo.info())

     purchase_year  purchase_month      product_category_name  target_demanda  \
0             2017               2  agro_industria_e_comercio               7   
39            2017               3  agro_industria_e_comercio               2   
137           2017               5  agro_industria_e_comercio               4   
194           2017               6  agro_industria_e_comercio               1   
253           2017               7  agro_industria_e_comercio               1   

     precio_promedio  es_black_friday  es_fin_de_semana  es_envio_gratis  \
0             32.120                0          0.428571              0.0   
39            40.995                0          0.000000              0.0   
137          394.985                0          0.750000              0.0   
194         1390.000                0          0.000000              0.0   
253         1180.000                0          0.000000              0.0   

     ventas_mes_pasado  tendencia_3_meses      mes_seno 

In [13]:
# Definimos la máscara temporal 
mask_test = (df_modelo['purchase_year'] == 2018) & (df_modelo['purchase_month'] >= 4)

# Tirar las columnas originales de año y mes de forma segura
columnas_a_tirar = ['purchase_year', 'purchase_month']
df_modelo = df_modelo.drop(columns=columnas_a_tirar)

df_train = df_modelo[~mask_test] # Pasado
df_test = df_modelo[mask_test]   # Futuro

print(f"Filas para Entrenamiento: {len(df_train)}")
print(f"Filas para Prueba: {len(df_test)}")

# Separamos X y Y (Target) para ambos conjuntos
# y
y_train = df_train['target_demanda']
y_test = df_test['target_demanda']

# X
X_train = df_train.drop(columns=['target_demanda'])
X_test = df_test.drop(columns=['target_demanda'])

print("Separación exitosa")

Filas para Entrenamiento: 822
Filas para Prueba: 326
Separación exitosa


In [ ]:





preprocesador = ColumnTransformer(
    transformers=[ # (nombre_paso, transformador, lista_de_columnas)
        # 'handle_unknown=ignore': si en Test aparece una categoría nueva (no en train), esta no rompe el código, la vuelve ceros.
        ('categorico', OneHotEncoder(handle_unknown='ignore'), ['product_category_name'])
    ],
    remainder='passthrough' # Deja pasar intactas el resto de columnas 
)

pipeline_rf = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', RandomForestRegressor(n_estimators=100, random_state=12345))
])

pipeline_rf.fit(X_train, y_train) # pasamos los datos enteros, el pipeline se encarga de las transformaciones

print("¡Modelo Baseline entrenado exitosamente!")

predicciones = pipeline_rf.predict(X_test) # Predicciones

# Calcular metricas 
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))
r2 = r2_score(y_test, predicciones)


print(" RESULTADOS DEL MODELO BASELINE (DATOS DE PRUEBA)")
print(f"MAE  (Error Absoluto Medio) : {mae:.2f} unidades")
print(f"RMSE (Error Cuadrático)     : {rmse:.2f} unidades")
print(f"R²   (Poder Explicativo)    : {r2:.4f} ({(r2*100):.1f}%)")

¡Modelo Baseline entrenado exitosamente!
 RESULTADOS DEL MODELO BASELINE (DATOS DE PRUEBA)
MAE  (Error Absoluto Medio) : 29.36 unidades
RMSE (Error Cuadrático)     : 56.28 unidades
R²   (Poder Explicativo)    : 0.9015 (90.1%)
